# FlyLM-3 · Visual Tiny Shakespeare → right-foreleg pen

An experimental successor to `flylm-2-training.ipynb`. A character is displayed to the eyes;
the retained MaleCNS circuit predicts the **next character as a pen trajectory**.

`character image → mapped R1–R6 → full recurrent MaleCNS → right foreleg MN activity → joint trajectories → pen tip`

Only photoreceptors receive external sensory current. Other sensory neurons remain in the
connectome but receive no external input. The entire `Traced` graph and its existing edges
are retained. No text embedding, token-to-neuron adapter, vocabulary projection, external LM,
proprioceptive input, reward injection, or shortcut from pixels to the pen controller is used.
Integer character indices are **dataset bookkeeping only**: `model.forward` accepts float images.

All implementation is in this notebook. Run top to bottom on Kaggle with Internet and a GPU.
The full graph is downloaded (~1.1 GB raw); allow several GB of RAM, disk cache, and GPU memory.
The initial full-graph Adam/backprop probe checks the configured batch and sequence length before training.
This notebook starts a new v3 experiment; v2 weights are incompatible.

**Scope of embodiment:** right-foreleg motor neurons are selected from MaleCNS annotations.
The pen is an engineered, differentiable two-link planar leg with an independent lift command.
A learned motor-only readout emits a fixed-length joint trajectory per character. It is an
open-loop kinematic surrogate, not a measured muscle model, physical fly simulation, or a claim
that the cited papers demonstrate writing. Link lengths, joint limits and ticks are dimensionless.
Generation displays and saves actual predicted pen traces; template recognition is only a readout.
No recognized character is substituted for the predicted trace fed back to the eyes.

**Hypothesis:** visual history propagated through the graph can reduce held-out next-character
trajectory error and template-derived negative log likelihood. Evaluate both: a low trajectory
loss alone can represent an unreadable average of possible next characters.

## 1. Configuration and runtime

Defaults are an execution experiment, not a language-quality promise. TBPTT carries detached
neural and retinal state through contiguous streams. Start with `train_updates=1000`, batch 2,
TBPTT 16 and 16 neural ticks per image; increasing ticks is much more expensive than v2.
Unknown activity defaults to spiking, as the effective v2 evidence cell did. Documented graded
types remain graded. `graded_control` is available for a matched control.

P100: install the official CUDA 12.6 build only if the installed torch lacks `sm_60`.
The probe checks actual sparse operations. A CPU runtime can run the small checks but is not
recommended for full-graph training. All settings are here; later cells do not override them.
The threshold center/radius (0.01/0.008) and graded lamina resting offset (0.30)
were selected using a short train-alphabet visual-response check: inherited v2 settings left
the foreleg silent under visual input. These dimensionless excitability choices are not
physiological estimates or evidence of language learning. No validation labels were used.

**Kaggle/P100 setup:** the compatibility probe imports torch in a separate process.
If replacement is necessary, install the exact `+cu126` wheels, verify `sm_60` again,
then continue in the same kernel. Installation itself does not require a restart.
Run this cell before importing torch elsewhere; an already-loaded incompatible native
CUDA library cannot safely be hot-reloaded. If the older notebook stopped at its
unconditional restart message, rerun this corrected cell: it can use the installed build.


In [ ]:
import importlib.util, subprocess, sys, json
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Select a runtime with PyTorch installed (Kaggle GPU recommended).')

def disk_cuda_info():
    # Import torch only in a child process until the installed build is ready.
    probe = subprocess.run([sys.executable, '-c',
        "import torch,json; print(json.dumps({'version':str(torch.__version__),'cap':list(torch.cuda.get_device_capability()) if torch.cuda.is_available() else None,'arch':torch.cuda.get_arch_list()}))"],
        capture_output=True, text=True, check=True)
    return json.loads(probe.stdout.strip().splitlines()[-1])

disk_info = disk_cuda_info()
print('Installed PyTorch:', disk_info)
if disk_info['cap'] == [6, 0] and 'sm_60' not in disk_info['arch']:
    print('Installing the CUDA 12.6 build with P100 support before importing torch.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade',
        'torch==2.10.0+cu126', 'torchvision==0.25.0+cu126', 'torchaudio==2.10.0+cu126',
        '--index-url', 'https://download.pytorch.org/whl/cu126'])
    importlib.invalidate_caches()
    disk_info = disk_cuda_info()
    print('After installation:', disk_info)
    if disk_info['cap'] != [6, 0] or 'sm_60' not in disk_info['arch']:
        raise RuntimeError('P100 support was not verified after installation. Check the pip output and GPU settings.')

# Like flyLM-2: a fresh kernel can continue immediately after installation.
# Do not remove/reload torch modules: native libraries already loaded cannot be safely swapped.
loaded_torch = sys.modules.get('torch')
if (disk_info['cap'] == [6, 0] and loaded_torch is not None
        and 'sm_60' not in loaded_torch.cuda.get_arch_list()):
    raise RuntimeError(
        'An incompatible torch build was already imported before this setup cell. '
        'Its live CUDA libraries cannot be safely replaced in place. '
        'Run this setup before any torch imports in a fresh kernel; no session restart '
        'is required when this setup runs first.'
    )
required = {'numpy':'numpy', 'pandas':'pandas', 'scipy':'scipy', 'pyarrow':'pyarrow',
            'tqdm':'tqdm', 'PIL':'pillow', 'matplotlib':'matplotlib'}
missing = [v for k,v in required.items() if importlib.util.find_spec(k) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import gc, io, hashlib, math, platform, random, shutil, time, urllib.request
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import scipy
from scipy import sparse
from scipy.sparse.csgraph import dijkstra
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties, findfont
from PIL import Image, ImageDraw

CFG = dict(seed=42, n_neurons=None, brain_steps=16, batch_size=2, tbptt=16,
    train_updates=1000, lr=0.001, neuron_mode='mixed', unknown_activity='spiking',
    spike_threshold_center=0.01, spike_threshold_radius=0.008, spike_reset=0.0,
    refractory_ticks=2, surrogate_slope=5.0, checkpoint_tokens=True,
    nt_confidence=0.8, use_nt_signs=True, edge_block=262144,
    tau_initial=(2.0,2.0,64.0), tau_min=(0.5,0.5,8.0), tau_max=(32.0,16.0,512.0),
    log_every=10, eval_every=100, save_every=100, eval_chunks=4,
    vram_fraction=0.90, image_size=48, pen_steps=128,
    retina_tau_ticks=4.0, retinal_blur=3, retinal_gain=1.0, retinal_half_saturation=0.02,
    # Intrinsic resting offset for graded lamina cells; no image information enters here.
    lamina_rest=0.30, link_lengths=(0.8,0.8),
    joint_low=(-1.8,0.2), joint_high=(1.0,2.8),
    ink_loss_weight=0.2, language_loss_weight=0.1, template_temperature=0.03)
WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()/'flylm3_work'
CACHE = WORK/'flylm_malecns_cache'
RUN_DIR = WORK/'flylm3_runs'/datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
CACHE.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_all(CFG['seed'])
VERSIONS = dict(python=platform.python_version(), torch=str(torch.__version__),
    numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__,
    pyarrow=pa.__version__, matplotlib=matplotlib.__version__)
assert all(CFG[k] > 0 for k in ['batch_size','tbptt','brain_steps','train_updates','pen_steps',
    'log_every','eval_every','save_every','eval_chunks','retina_tau_ticks','template_temperature'])
assert 0 < CFG['vram_fraction'] < 1
assert isinstance(CFG['retinal_blur'],int) and CFG['retinal_blur']>0 and CFG['retinal_blur']%2==1
assert CFG['image_size']>1
r = torch.tensor([0,1,2], device=device)
c = torch.tensor([1,0], device=device)
w = torch.tensor([2.,3.], device=device)
torch.testing.assert_close(torch.sparse.mm(torch.sparse_csr_tensor(r,c,w,size=(2,2)),
    torch.ones(2,1,device=device)), torch.tensor([[2.],[3.]],device=device))
print(VERSIONS, 'device:', device, 'output:', RUN_DIR)

## 2. Tiny Shakespeare and visual/pen alphabet

Use the raw character corpus referenced by nanoGPT's **shakespeare_char/prepare.py**, not its
GPT-2-tokenized `shakespeare` preparation. Preserve case, punctuation, spaces and newlines.
The first 90% of characters train; the last 10% validate. No pair crosses the boundary.
The alphabet is defined from train only and validation must contain no unseen character.

DejaVu Sans Mono outlines define reproducible **stroke-tracing targets**, not natural handwriting.
An outline contour starts with a pen-up move, then closes with ink. Allocate 128 samples across
contours by arc length. Both images and target trajectories are generated here, without a
handwriting dataset or recognizer. All characters share one scale and baseline.
Space is displayed/written as `␣` and newline as `↵` so they are visibly distinguishable;
these explicit control marks are decoded to whitespace in the text log. There is no EOS/BOS token.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
data_path = CACHE/'tinyshakespeare_input.txt'
if not data_path.exists():
    with urllib.request.urlopen(DATA_URL, timeout=60) as response:
        payload = response.read()
    tmp = data_path.with_suffix('.part'); tmp.write_bytes(payload); tmp.replace(data_path)
payload = data_path.read_bytes()
assert hashlib.sha256(payload).hexdigest() == '86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed', 'Tiny Shakespeare content mismatch'
text = payload.decode('utf-8')  # no whitespace normalization
assert len(text) == 1115394, 'Unexpected Tiny Shakespeare version/partial download.'
split = int(0.9*len(text))
train_text, val_text = text[:split], text[split:]
chars = sorted(set(train_text)); char_to_index = {ch:i for i,ch in enumerate(chars)}
assert set(val_text) <= set(chars)
train_ids = np.array([char_to_index[ch] for ch in train_text], dtype=np.int64)
val_ids = np.array([char_to_index[ch] for ch in val_text], dtype=np.int64)
DATA_SPEC = dict(url=DATA_URL, sha256=hashlib.sha256(payload).hexdigest(),
    train_characters=len(train_text), validation_characters=len(val_text), alphabet=chars,
    preparation='https://github.com/Lucid-Frameworks/nanoGPT/blob/master/data/shakespeare_char/prepare.py')
font_path = Path(findfont(FontProperties(family='DejaVu Sans Mono'), fallback_to_default=False))
FONT_SPEC = dict(name=font_path.name, sha256=hashlib.sha256(font_path.read_bytes()).hexdigest(),
    whitespace={'space':'␣','newline':'↵'}, target='outline_contours_arc_length')
font = FontProperties(fname=str(font_path))

def glyph_contours(ch):
    visible = {' ':'␣', '\n':'↵'}.get(ch,ch)
    polygons = TextPath((0,0), visible, size=1, prop=font).to_polygons()
    result = []
    for p in polygons:
        if len(p) < 3: continue
        if not np.allclose(p[0],p[-1]): p = np.vstack([p,p[0]])
        if np.linalg.norm(np.diff(p,axis=0),axis=1).sum() > 1e-8: result.append(p)
    if not result: raise ValueError(f'No drawable contours: {ch!r}')
    return result
all_contours = [glyph_contours(ch) for ch in chars]
all_xy = np.concatenate([p for contours in all_contours for p in contours])
lo, hi = all_xy.min(0), all_xy.max(0)
scale = 0.8 / max(hi-lo)
center = (lo+hi)/2

def make_trace(contours, steps):
    lengths = np.array([np.linalg.norm(np.diff(p,axis=0),axis=1).sum() for p in contours])
    # Each contour gets >=3 ink segments plus one pen-up start.
    assert steps >= 4*len(contours)
    extra = steps-4*len(contours)
    alloc = np.floor(extra*lengths/lengths.sum()).astype(int)+4
    for i in np.argsort(-(extra*lengths/lengths.sum()%1))[:steps-alloc.sum()]: alloc[i] += 1
    trace = []
    for poly, count in zip(contours, alloc):
        dist = np.r_[0,np.cumsum(np.linalg.norm(np.diff(poly,axis=0),axis=1))]
        keep = np.r_[True,np.diff(dist)>1e-10]
        poly,dist = poly[keep],dist[keep]
        sample = np.linspace(0,dist[-1],count)
        xy = np.stack([np.interp(sample,dist,poly[:,j]) for j in range(2)],axis=-1)
        xy = (xy-center)*scale+np.array([0.8,0.0])
        ink = np.ones(count); ink[0]=0
        trace.append(np.column_stack([xy,ink]))
    return np.concatenate(trace).astype(np.float32)

def forward_kinematics(q, lengths):
    q1,q2 = q.unbind(-1); l1,l2 = lengths
    return torch.stack([l1*q1.cos()+l2*(q1+q2).cos(),
                        l1*q1.sin()+l2*(q1+q2).sin()],dim=-1)

def inverse_kinematics(xy, lengths):
    x,y = xy.unbind(-1); l1,l2 = lengths
    cos2 = (x*x+y*y-l1*l1-l2*l2)/(2*l1*l2)
    assert torch.all(cos2.abs() < 1), 'Glyph outside reachable workspace'
    q2 = cos2.acos()
    q1 = torch.atan2(y,x)-torch.atan2(l2*q2.sin(),l1+l2*q2.cos())
    return torch.stack([q1,q2],dim=-1)

def rasterize_trace(trace, size=None):
    size = CFG['image_size'] if size is None else size
    a = np.asarray(trace); oversample=3
    canvas = Image.new('L',(size*oversample,size*oversample),0)
    draw = ImageDraw.Draw(canvas)
    # Fixed display frame, x in [0.25,1.35], y in [-0.55,0.55].
    xy = np.column_stack([(a[:,0]-.25)/1.1, (.55-a[:,1])/1.1])*(size*oversample-1)
    for j in range(1,len(a)):
        if a[j,2] >= 0.5:
            draw.line([tuple(xy[j-1]),tuple(xy[j])],fill=255,width=oversample)
    return np.asarray(canvas.resize((size,size),Image.Resampling.LANCZOS),dtype=np.float32)/255

target_trace_cpu = torch.from_numpy(np.stack([make_trace(p,CFG['pen_steps']) for p in all_contours]))
target_joints_cpu = inverse_kinematics(target_trace_cpu[...,:2],CFG['link_lengths'])
torch.testing.assert_close(forward_kinematics(target_joints_cpu,CFG['link_lengths']),
    target_trace_cpu[...,:2],atol=2e-6,rtol=2e-6)
assert torch.all(target_joints_cpu > torch.tensor(CFG['joint_low']))
assert torch.all(target_joints_cpu < torch.tensor(CFG['joint_high']))
glyph_images_cpu = torch.from_numpy(np.stack([rasterize_trace(t) for t in target_trace_cpu.numpy()]))[:,None]
assert len(set(hashlib.sha256(x.numpy().tobytes()).hexdigest() for x in glyph_images_cpu)) == len(chars)
target_trace = target_trace_cpu.to(device); target_joints = target_joints_cpu.to(device)
glyph_images = glyph_images_cpu.to(device)
print(DATA_SPEC, FONT_SPEC)
fig,axes = plt.subplots(2,8,figsize=(12,3))
for ax,i in zip(axes.flat,np.linspace(0,len(chars)-1,16,dtype=int)):
    ax.imshow(glyph_images_cpu[i,0],cmap='gray',vmin=0,vmax=1); ax.set_title(repr(chars[i])); ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Full MaleCNS graph

Preserve v2's official v1.0 download, source hashing, `status == 'Traced'` selection,
two-pass CSR cache, all weak/self edges and summed duplicate contacts.
CSR orientation is **post row, pre column**. The readout restriction does not prune the graph.

In [ ]:
BASE_URL = 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/'
SOURCES = {
    'annotations': ('body-annotations-male-cns-v1.0-minconf-0.5.feather', 14483314),
    'nt': ('body-neurotransmitters-male-cns-v1.0.feather', 43282834),
    'edges': ('connectome-weights-male-cns-v1.0-minconf-0.5.feather', 1051241946),
}

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()

def download_file(name, expected_size):
    target = CACHE / name
    if not target.exists() or target.stat().st_size != expected_size:
        if shutil.disk_usage(CACHE).free < expected_size + 512 * 1024**2:
            raise RuntimeError('데이터 다운로드에 필요한 디스크 공간이 부족합니다.')
        partial = target.with_suffix(target.suffix + '.part')
        with urllib.request.urlopen(BASE_URL + name, timeout=60) as response:
            with partial.open('wb') as f, tqdm(total=expected_size, unit='B', unit_scale=True, desc=name) as bar:
                while True:
                    chunk = response.read(8 * 1024**2)
                    if not chunk:
                        break
                    f.write(chunk)
                    bar.update(len(chunk))
        if partial.stat().st_size != expected_size:
            raise RuntimeError(f'파일 크기 불일치: {name}. v1.0 파일/다운로드를 확인하세요.')
        partial.replace(target)
    return target

files = {key: download_file(*spec) for key, spec in SOURCES.items()}
source_manifest = {key: dict(url=BASE_URL + p.name, bytes=p.stat().st_size, sha256=sha256_file(p))
                   for key, p in files.items()}
print({key: entry['bytes'] for key, entry in source_manifest.items()})

In [ ]:
def require_columns(names, required, label):
    absent = set(required)-set(names)
    if absent:
        raise ValueError(f'{label}: missing {sorted(absent)}; found {list(names)}')

def map_ids(sorted_ids, values):
    positions = np.searchsorted(sorted_ids, values)
    safe = positions.clip(0, len(sorted_ids)-1)
    return positions, (positions < len(sorted_ids)) & (sorted_ids[safe] == values)

annotation = feather.read_table(files['annotations']).to_pandas()
require_columns(annotation.columns, ['bodyId','status','superclass'], 'annotations')
neurons = annotation.loc[annotation.status == 'Traced'].sort_values('bodyId').set_index('bodyId')
assert neurons.index.is_unique and not neurons.index.hasnans
body_ids = neurons.index.to_numpy(np.int64)
n = len(body_ids)
CFG['n_neurons'] = n
assert n > 100_000, f'전체 Traced 데이터인지 확인하세요: {n}'
print('Actual full Traced neuron count:', n)
print(neurons['superclass'].value_counts(dropna=False).to_string())

def full_edge_batches():
    with pa.memory_map(str(files['edges']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body_pre','body_post','weight'], 'edges')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            s, sm = map_ids(body_ids, batch.column('body_pre').to_numpy(zero_copy_only=False))
            d, dm = map_ids(body_ids, batch.column('body_post').to_numpy(zero_copy_only=False))
            keep = sm & dm
            if keep.any():
                w = batch.column('weight').to_numpy(zero_copy_only=False)[keep]
                if not np.isfinite(w).all() or (w <= 0).any():
                    raise ValueError('Invalid count')
                yield s[keep].astype(np.int32), d[keep].astype(np.int32), w.astype(np.int64)

extract_spec = dict(version=3, selection='all_status_Traced_no_component_filter',
                    orientation='post_row_pre_column', n=n,
                    source_sha256={k:v['sha256'] for k,v in source_manifest.items()})
cache_key = hashlib.sha256(json.dumps(extract_spec, sort_keys=True).encode()).hexdigest()[:16]
full_cache = CACHE / f'full_{cache_key}'
full_cache.mkdir(exist_ok=True)
manifest_path = full_cache / 'complete.json'
if not manifest_path.exists():
    degree = np.zeros(n, np.int64)
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 1: full degrees'):
        degree += np.bincount(d, minlength=n)
    raw_edges = int(degree.sum())
    if raw_edges == 0:
        raise ValueError('No full-neuron edges')
    required_disk = raw_edges*24 + 512*1024**2
    if shutil.disk_usage(full_cache).free < required_disk:
        raise RuntimeError(f'CSR 구축 디스크가 부족합니다. 약 {required_disk/2**30:.1f} GiB 여유가 필요합니다.')
    crow = np.concatenate(([0], np.cumsum(degree)))
    col = np.lib.format.open_memmap(full_cache/'src.npy', mode='w+', dtype=np.int32, shape=(raw_edges,))
    count = np.lib.format.open_memmap(full_cache/'count.npy', mode='w+', dtype=np.int64, shape=(raw_edges,))
    cursor = crow[:-1].copy()
    for s, d, w in tqdm(full_edge_batches(), desc='Pass 2: write full CSR'):
        order = np.argsort(d, kind='stable')
        sd = d[order]
        starts = np.r_[0, np.flatnonzero(sd[1:] != sd[:-1])+1]
        sizes = np.diff(np.r_[starts, len(sd)])
        local = np.arange(len(sd))-np.repeat(starts, sizes)
        positions = cursor[sd]+local
        col[positions], count[positions] = s[order], w[order]
        cursor += np.bincount(d, minlength=n)
    assert np.array_equal(cursor, crow[1:])
    col.flush()
    count.flush()
    full = sparse.csr_matrix((count, col, crow), shape=(n,n), copy=False)
    full.sum_duplicates()
    full.sort_indices()
    nnz = int(full.nnz)
    # 중복 병합 후 유효한 배열 구간과 행 포인터를 명시적으로 저장한다.
    col[:nnz], count[:nnz] = full.indices, full.data
    col.flush()
    count.flush()
    np.save(full_cache/'crow.npy', full.indptr.astype(np.int64))
    np.save(full_cache/'body_ids.npy', body_ids)
    manifest_path.write_text(json.dumps(dict(spec=extract_spec, raw_edges=raw_edges, nnz=nnz)), encoding='utf-8')
    del full, col, count, cursor, degree, crow
    gc.collect()
full_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert full_manifest['spec'] == extract_spec
crow = np.load(full_cache/'crow.npy', mmap_mode='r')
src = np.load(full_cache/'src.npy', mmap_mode='r')[:full_manifest['nnz']]
counts = np.load(full_cache/'count.npy', mmap_mode='r')[:full_manifest['nnz']]
assert int(crow[-1]) == len(src) == len(counts)
assert np.array_equal(body_ids, np.load(full_cache/'body_ids.npy'))
graph_data = dict(body_ids=body_ids, crow=crow, src=src, counts=counts)
print(dict(neurons=n, edges=len(src), retained='all Traced-to-Traced pairs'))

## 4. Eyes and right foreleg: audited anatomical ports

The retinal mapping adapts [doomfly prepare.py](https://github.com/nftechie/doomfly/blob/main/doom/prepare.py):
for each R1–R6 receptor, aggregate **all contacts** onto hex-annotated L1/L2/L3 and choose its
highest-contact column. Map each eye to an overlapping screen viewport. This estimates display
coordinates; it is not calibrated eye optics. Unmapped receptors remain present without pixels.
Pixel luminance is sRGB-linearized, spatially averaged over a 3×3 pixel footprint,
low-pass filtered, then saturated as in doomfly, with gain
and time expressed in v2's dimensionless model units. The finite sampling footprint is
an engineered anti-aliasing choice, not measured ommatidial optics. Graded lamina resting excitation is
an explicit image-independent model offset, not an additional sensory port.

Right foreleg selector: `vnc_motor`, `subclass == fl`, `somaSide == R`, `somaNeuromere == T1`.
This uses the downloaded MaleCNS IDs, not FANC IDs or a generic motor fallback. Export the
selected IDs and labels for inspection. Fail if no mapped retina, no foreleg motor, or no path.

In [ ]:
KNOWN_NT = {'acetylcholine', 'gaba', 'glutamate', 'histamine', 'dopamine', 'serotonin', 'octopamine'}

def attach_nt(body_ids):
    selected_parts = []
    wanted = np.sort(body_ids)
    with pa.memory_map(str(files['nt']), 'r') as source:
        reader = pa.ipc.open_file(source)
        require_columns(reader.schema.names, ['body', 'ground_truth', 'predicted_nt',
                                              'predicted_nt_confidence', 'consensus_nt'], 'neurotransmitters')
        for i in range(reader.num_record_batches):
            batch = reader.get_batch(i)
            _, keep = map_ids(wanted, batch.column('body').to_numpy(zero_copy_only=False))
            if keep.any():
                selected_parts.append(batch.filter(pa.array(keep)).to_pandas())
    if not selected_parts:
        raise RuntimeError('선택된 body와 전달물질 테이블의 join이 비어 있습니다.')
    table = pd.concat(selected_parts, ignore_index=True)
    if table['body'].duplicated().any():
        raise ValueError('Duplicate NT body IDs: explicit aggregation is required')
    table = table.set_index('body').reindex(body_ids)
    truth = table['ground_truth'].fillna('').str.strip().str.lower()
    predicted = table['predicted_nt'].fillna('').str.strip().str.lower()
    confidence = pd.to_numeric(table['predicted_nt_confidence'], errors='coerce')
    trusted = truth.isin(KNOWN_NT)
    confident = predicted.isin(KNOWN_NT) & confidence.ge(CFG['nt_confidence']) & confidence.le(1)
    effective = np.where(trusted, truth, np.where(confident, predicted, 'unknown'))
    origin = np.where(trusted, 'ground_truth', np.where(confident, 'predicted_nt', 'unknown'))
    table['effective_nt'] = effective
    table['effective_source'] = origin
    table.index.name = 'body'
    return table



require_columns(neurons.columns, ['type','rootSide','assignedOlHex1','assignedOlHex2',
    'subclass','somaSide','somaNeuromere'], 'visual/motor annotations')
receptor_mask = neurons['type'].eq('R1-R6').to_numpy()
anchor_mask = (neurons['type'].isin(['L1','L2','L3']) &
    neurons['assignedOlHex1'].notna() & neurons['assignedOlHex2'].notna()).to_numpy()
votes = {}
for post in np.flatnonzero(anchor_mask):
    left,right = int(crow[post]),int(crow[post+1])
    ss,ww = src[left:right],counts[left:right]
    h = (float(neurons.iloc[post].assignedOlHex1),float(neurons.iloc[post].assignedOlHex2))
    for pre,w in zip(ss[receptor_mask[ss]],ww[receptor_mask[ss]]):
        d = votes.setdefault(int(pre),{}); d[h] = d.get(h,0)+int(w)
input_nodes = np.array(sorted(votes),dtype=np.int64)
assert len(input_nodes)>0, 'No mapped R1-R6 receptors'
hexes = np.array([max(votes[i], key=lambda h:(votes[i][h],h)) for i in input_nodes])
confidence = np.array([max(votes[i].values())/sum(votes[i].values()) for i in input_nodes])
xy = np.column_stack([hexes[:,0]-.5*hexes[:,1],np.sqrt(3)/2*hexes[:,1]])
sides = neurons.iloc[input_nodes].rootSide.to_numpy()
assert set(sides) == {'L','R'}, 'Retinal root sides must be explicitly L/R'
uv = np.zeros_like(xy,dtype=np.float32)
for side in ['L','R']:
    mask = sides == side; points=xy[mask]; span=np.ptp(points,axis=0)
    assert (span>0).all()
    z=(points-points.min(0))/span
    uv[mask,0] = .6*z[:,0] if side=='L' else .4+.6*(1-z[:,0])
    uv[mask,1] = 1-z[:,1]
motor_mask = (neurons.superclass.eq('vnc_motor') & neurons.subclass.eq('fl') &
    neurons.somaSide.eq('R') & neurons.somaNeuromere.eq('T1')).to_numpy()
output_nodes = np.flatnonzero(motor_mask).astype(np.int64)
lamina_nodes = np.flatnonzero(neurons['type'].isin(['L1','L2','L3','L5'])).astype(np.int64)
assert len(output_nodes)>0, 'No right foreleg motor annotations; no fallback is permitted'
assert neurons.iloc[input_nodes].superclass.eq('ol_sensory').all()
assert not np.intersect1d(input_nodes,output_nodes).size
nt_table = attach_nt(body_ids)
# Histamine is explicitly inhibitory in this visual proxy; other uncertain signs retain v2 freedom.
node_sign = nt_table.effective_nt.map({'acetylcholine':1.,'gaba':-1.,'histamine':-1.}).fillna(0).to_numpy(np.float32)
if not CFG['use_nt_signs']: node_sign[:] = 0
edge_sign = node_sign[src]
adj_post_pre = sparse.csr_matrix((np.ones(len(src),np.uint8),src,crow),shape=(n,n))
from_retina = dijkstra(adj_post_pre.T.tocsr(),directed=True,indices=input_nodes,min_only=True,unweighted=True)
motor_dist = from_retina[output_nodes]
assert np.isfinite(motor_dist).any(), 'No directed retinal → right-foreleg path'
GRAPH_SPEC = dict(neurons=n, edges=len(src), mapped_receptors=len(input_nodes),
    unmapped_receptors=int(receptor_mask.sum()-len(input_nodes)), right_foreleg_motors=len(output_nodes),
    reachable_motors=int(np.isfinite(motor_dist).sum()), shortest_motor_path=float(np.min(motor_dist)),
    median_retina_confidence=float(np.median(confidence)),
    foreleg_selector='vnc_motor & fl & R somaSide & T1 somaNeuromere')
neurons.iloc[output_nodes].to_csv(RUN_DIR/'right_foreleg_motors.csv')
pd.DataFrame(dict(bodyId=body_ids[input_nodes],u=uv[:,0],v=uv[:,1],confidence=confidence)).to_csv(
    RUN_DIR/'retinal_projection.csv',index=False)
print(GRAPH_SPEC)
print(neurons.iloc[output_nodes][['type','subclass','somaSide','somaNeuromere']].to_string())
del adj_post_pre, votes, annotation
gc.collect()

def dynamics_types(frame):
    labels = pd.Series('', index=frame.index, dtype=object)
    for column in ['type', 'class', 'superclass']:
        if column not in frame:
            continue
        values = frame[column].fillna('').astype(str).str.strip()
        valid = ~values.str.lower().isin(['', 'none', 'nan', 'unknown', 'tbc', 'untyped'])
        use = labels.eq('') & valid
        labels.loc[use] = column + ':' + values.loc[use]
    labels = labels.mask(labels.eq(''), 'unknown')
    codes, names = pd.factorize(labels, sort=True)
    return codes.astype(np.int64), list(names)

type_ids, type_names = dynamics_types(neurons)
print('Dynamics groups:', len(type_names), 'type-annotated neurons:',
      sum(type_names[i].startswith('type:') for i in type_ids))

## 5. Mixed dynamics inherited from v2

Retain the exact-type evidence table and surrogate-spike update from v2. Unmatched cell types
are **assumed**, not measured. Add L1/L2/L3/L5 as a graded visual-model choice, separately
identified from the inherited electrophysiology evidence. The photoreceptors remain graded.
This is gradient-based task optimization, not biological synaptic plasticity.

In [ ]:
# 문헌의 세포 유형을 MaleCNS의 정확한 type 문자열에 연결하는 작은 근거집이다.
EVIDENCE_VERSION = 'flylm3_activity_evidence_v1'


ACTIVITY_EVIDENCE = [
    dict(type_name='H2', activity='spiking',
         evidence_level='recorded_type_transfer', species='Drosophila melanogaster',
         preparation='성체; H2 전세포 기록', sex='이 근거집에서는 성별을 확정하지 않음',
         source='https://doi.org/10.1038/s41586-025-09276-5',
         locator='H2 electrophysiology; 국소 운동 자극에 대한 스파이크 기록',
         finding='H2의 활동전위를 직접 기록하였다.',
         mapping='MaleCNS type H2와 문헌의 H2를 세포 유형 수준에서 대응시킨다.',
         limitation='다른 개체의 유형 근거이며 MaleCNS 개체의 직접 측정이 아니다. 역치·불응기는 옮겨오지 않는다.'),
    dict(type_name='DM1_lPN', activity='spiking',
         evidence_level='recorded_type_transfer', species='Drosophila melanogaster',
         preparation='성체 2–10일; DM1 투사뉴런 전세포 기록', sex='female',
         source='https://doi.org/10.1523/JNEUROSCI.0764-09.2009',
         locator='Methods: NP5221-Gal4 DM1 PNs; Fig. 5B',
         finding='DM1 투사뉴런에서 실제 활동전위를 기록하였다.',
         mapping='DM1 uniglomerular PN을 MaleCNS DM1_lPN에 대응시키는 해부학적 추론이다.',
         limitation='암컷의 DM1 유형 근거를 수컷 대응 유형으로 전이한다. 모든 후각 투사뉴런으로 확장하지 않는다.'),
    *[dict(type_name=name, activity='graded',
           evidence_level='recorded_type_transfer', species='Drosophila melanogaster',
           preparation='성체 수컷; 생체 내 HS 전세포 기록', sex='male',
           source='https://doi.org/10.1038/s41467-024-53173-w',
           locator='Introduction; Fig. 5B–D; Methods: male flies',
           finding='HS의 운동 반응은 연속 막전위 변화로 다룬다. HSN/HSE/HSS 기록을 구별한다.',
           mapping=f'MaleCNS type {name}를 문헌의 동명 HS 유형에만 대응시킨다.',
           limitation='주된 연속 신호를 근사한다. 모든 조건에서 재생성 사건이 불가능하다는 주장이 아니다. 전기 시냅스는 추가하지 않는다.')
      for name in ('HSN', 'HSE', 'HSS')],
    dict(type_name='R1-R6', activity='graded',
         evidence_level='recorded_family_transfer', species='Drosophila melanogaster',
         preparation='생체 내 R1–R6 광수용체 세포 내 전압 기록', sex='이 근거집에서는 성별을 확정하지 않음',
         source='https://doi.org/10.3791/54142',
         locator='Introduction; Representative Results: R1–R6 voltage responses',
         finding='R1–R6의 빛 반응을 연속 전압 신호로 기록한다.',
         mapping='MaleCNS의 통합 라벨 R1-R6에만 적용한다. R7/R8에는 전파하지 않는다.',
         limitation='기록된 세포군의 주된 신호 방식을 전이한다. 개별 R1–R6의 매개변수 차이는 보존하지 못한다.'),
]


def assign_activity_modes(frame, evidence, unknown_mode='spiking'):
    if unknown_mode not in ('graded', 'spiking'):
        raise ValueError('미확인 뉴런의 기본값은 graded 또는 spiking이어야 한다.')
    if not frame.index.is_unique or 'type' not in frame:
        raise ValueError('고유 bodyId 인덱스와 type 열이 필요하다.')

    lookup = {}
    for entry in evidence:
        name = entry['type_name']
        if not isinstance(name, str) or not name or name != name.strip() or name in lookup:
            raise ValueError(f'중복 또는 잘못된 근거 유형: {name}')
        if entry['activity'] not in ('graded', 'spiking'):
            raise ValueError(f'잘못된 신호 방식: {name}')
        for key in ('source', 'locator', 'finding', 'mapping', 'limitation',
                    'evidence_level', 'species', 'preparation', 'sex'):
            if not isinstance(entry.get(key), str) or not entry[key].strip():
                raise ValueError(f'근거 필드 누락: {name}, {key}')
        if not entry['source'].startswith('https://'):
            raise ValueError(f'근거 출처에는 HTTPS 링크가 필요하다: {name}')
        lookup[name] = entry

    types = frame['type'].fillna('').astype(str).str.strip()

    # 근거 코드: 0은 미확인, 1은 연속 신호 근거, 2는 스파이크 근거이다.
    codes = types.map({
        k: 2 if v['activity'] == 'spiking' else 1
        for k, v in lookup.items()
    })
    codes = codes.fillna(0).to_numpy(dtype=np.uint8)

    records = pd.DataFrame({
        'bodyId': frame.index.to_numpy(),
        'type': types.to_numpy(),
        'evidence_code': codes,
    })
    records['evidence_activity'] = np.take(['unknown', 'graded', 'spiking'], codes)

    # 미확인 유형에만 기본값을 적용하고, 문헌상 연속 유형은 그대로 유지한다.
    records['assigned_activity'] = np.where(
        codes == 0, unknown_mode,
        np.where(codes == 2, 'spiking', 'graded'),
    )

    for key in ('source', 'locator', 'finding', 'mapping', 'limitation',
                'evidence_level', 'species', 'preparation', 'sex'):
        records[key] = types.map({
            k: v[key] for k, v in lookup.items()
        }).fillna('').to_numpy()

    records.loc[codes == 0, 'limitation'] = (
        f'이 근거집에서 미확인; {unknown_mode} 동역학을 가정한다. 직접 측정 근거가 아니다.'
    )
    records.loc[codes == 0, 'evidence_level'] = 'unknown_default'

    by_type = records.groupby(
        ['type', 'evidence_activity', 'assigned_activity'], dropna=False
    ).size()

    summary = dict(
        neurons=len(frame),
        evidence_supported_neurons=int((codes != 0).sum()),
        evidence_spiking_neurons=int((codes == 2).sum()),
        evidence_graded_neurons=int((codes == 1).sum()),
        unknown_default_neurons=int((codes == 0).sum()),
        evidence_fraction=float((codes != 0).mean()),
        unmatched_evidence_types=sorted(set(lookup) - set(types)),
        unknown_default=unknown_mode,
    )

    # 문헌 근거 코드와 실제 실행할 동역학 코드를 구분한다.
    # 미확인 뉴런의 evidence_code는 0으로 남지만, 실행 코드는 기본값을 따른다.
    assigned_codes = np.where(
        records['assigned_activity'].to_numpy() == 'spiking', 2, 1
    ).astype(np.uint8)

    return assigned_codes, records, by_type.reset_index(name='neurons'), summary


ACTIVITY_EVIDENCE.extend([
    dict(type_name=name,activity='graded',evidence_level='visual_model_assumption',
         species='Drosophila melanogaster',preparation='model transfer',sex='not specified',
         source='https://doi.org/10.1038/s41586-024-07939-3',locator='visual network graded model',
         finding='Use graded visual dynamics for the lamina proxy.',
         mapping='Exact MaleCNS type label',limitation='Model choice, not recording from this specimen.')
    for name in ['L1','L2','L3','L5']])
dynamics_ids, activity_annotations, activity_types, activity_coverage = assign_activity_modes(
    neurons, ACTIVITY_EVIDENCE, CFG['unknown_activity']
)

# graded_control에서는 배정 결과와 관계없이 모든 뉴런에 연속 동역학을 적용한다.
activity_annotations['effective_activity'] = np.where(
    (dynamics_ids == 2) & (CFG['neuron_mode'] == 'mixed'), 'spiking', 'graded'
)
activity_coverage['effective_spiking_neurons'] = int(
    ((dynamics_ids == 2) & (CFG['neuron_mode'] == 'mixed')).sum()
)
activity_coverage['neuron_mode'] = CFG['neuron_mode']

dynamics_spec = dict(
    version=EVIDENCE_VERSION,
    mode=CFG['neuron_mode'],
    unknown_activity=CFG['unknown_activity'],
    evidence=ACTIVITY_EVIDENCE,
    state_layout=['membrane', 'synaptic_signal', 'adaptation', 'refractory_ticks'],
    time_unit='simulation_tick_not_ms',
    parameters='engineering_assumptions',
)

dynamics_json = json.dumps(
    dynamics_spec, ensure_ascii=False, sort_keys=True, indent=2
)
dynamics_sha256 = hashlib.sha256(dynamics_json.encode('utf-8')).hexdigest()

activity_annotations.to_csv(RUN_DIR/'neuron_activity_annotations.csv', index=False)
activity_types.to_csv(RUN_DIR/'activity_types.csv', index=False)
(RUN_DIR/'dynamics_evidence.json').write_bytes(dynamics_json.encode('utf-8'))
(RUN_DIR/'activity_coverage.json').write_text(
    json.dumps(activity_coverage, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print('신경 활동 근거 범위:', json.dumps(activity_coverage, ensure_ascii=False, indent=2))
print(activity_types.loc[activity_types.evidence_activity != 'unknown'].to_string(index=False))

if activity_coverage['unmatched_evidence_types']:
    print(
        '주의: 원본 type과 일치하지 않아 적용되지 않은 근거:',
        activity_coverage['unmatched_evidence_types'],
    )

if CFG['neuron_mode'] == 'mixed' and not np.any(dynamics_ids == 2):
    raise ValueError('스파이크 동역학이 배정된 뉴런이 없다. 유형 배정과 기본값을 확인하세요.')

print(
    f"미확인 유형의 배정 기본값: {CFG['unknown_activity']}. "
    '이는 문헌상 신호 방식이 확인되었다는 뜻이 아니다.'
)
print('graded_control 모드에서는 이 배정과 관계없이 모든 뉴런이 연속 동역학으로 실행된다.')

## 6. Differentiable visual circuit and leg pen

Neural state is `(v, signal, adaptation, refractory)` plus retinal low-pass luminance.
The pen head sees only right-foreleg motor signals. It maps their normalized activity into
`128 × (joint1, joint2, lift)` commands, with bounded angles, forward kinematics and pen contact.
It does not read images, labels, targets, other neurons, or a separate recurrent language model.
Its temporal trajectory basis is learned in the small output adapter rather than simulated
as 128 further full-connectome steps. No force, friction or muscle recruitment is modeled.

Loss combines pen-tip MSE, contact BCE, and a small next-character loss derived from distances
between **predicted traces** and fixed template traces. There is no trainable vocabulary head.
That distribution is an engineered template likelihood, not standard LM perplexity or OCR.

In [ ]:
class SparseEdges(torch.autograd.Function):
    @staticmethod
    def forward(ctx, h, weights, rowptr, col, dst, block):
        matrix = torch.sparse_csr_tensor(rowptr, col, weights, size=(h.shape[1], h.shape[1]),
                                         device=h.device, dtype=h.dtype)
        ctx.save_for_backward(h, weights, rowptr, col, dst)
        ctx.block = block
        return torch.sparse.mm(matrix, h.T.contiguous()).T.contiguous()

    @staticmethod
    def backward(ctx, grad_output):
        h, weights, rowptr, col, dst = ctx.saved_tensors
        grad_h = torch.zeros_like(h) if ctx.needs_input_grad[0] else None
        grad_w = torch.empty_like(weights) if ctx.needs_input_grad[1] else None
        for lo in range(0, col.numel(), ctx.block):
            hi = min(col.numel(), lo+ctx.block)
            sources, targets = col[lo:hi], dst[lo:hi]
            g = grad_output.index_select(1, targets)
            if grad_w is not None:
                grad_w[lo:hi] = (g*h.index_select(1, sources)).sum(0)
            if grad_h is not None:
                grad_h.index_add_(1, sources, g*weights[lo:hi])
        return grad_h, grad_w, None, None, None, None

class SurrogateSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, voltage_gap, slope):
        ctx.save_for_backward(voltage_gap)
        ctx.slope = slope
        return (voltage_gap >= 0).to(voltage_gap.dtype)

    @staticmethod
    def backward(ctx, grad_output):
        (gap,) = ctx.saved_tensors
        # 순전파는 이산 사건이며, 역전파에만 빠른 시그모이드의 대리 미분을 쓴다.
        return grad_output / (1 + ctx.slope * gap.abs()).square(), None


def mixed_neuron_tick(v, signal, slow, refractory, current, alpha, threshold,
                      adaptation, spiking_mask, reset_voltage, refractory_ticks, surrogate_slope):
    candidate = v + alpha[0] * (-v + current - adaptation * slow)
    blocked = spiking_mask & (refractory > 0)
    candidate = torch.where(blocked, torch.full_like(candidate, reset_voltage), candidate)
    spikes = SurrogateSpike.apply(candidate - threshold, surrogate_slope)
    spikes = spikes * (spiking_mask & ~blocked).to(candidate.dtype)
    graded = torch.relu(torch.tanh(candidate - threshold))
    release = torch.where(spiking_mask, spikes, graded)
    # 리셋·불응기 사건의 이산 결정에는 별도 대리 미분을 적용하지 않는다.
    fired = spikes.detach() > 0
    v = torch.where(fired, torch.full_like(candidate, reset_voltage), candidate)
    refractory = torch.where(fired, torch.full_like(refractory, float(refractory_ticks)),
                             torch.clamp(refractory - 1, min=0))
    refractory = torch.where(spiking_mask, refractory, torch.zeros_like(refractory)).detach()
    signal = signal + alpha[1] * (release - signal)
    slow = slow + alpha[2] * (release - slow)
    return v, signal, slow, refractory, spikes


def validate_dynamics_config(cfg):
    if cfg['neuron_mode'] not in ('mixed', 'graded_control'):
        raise ValueError('neuron_mode은 mixed 또는 graded_control이어야 한다.')
    if cfg['unknown_activity'] not in ('graded', 'spiking'):
        raise ValueError('미확인 뉴런의 기본값은 graded 또는 spiking이어야 한다.')
    for key in ('spike_threshold_center', 'spike_threshold_radius', 'spike_reset', 'surrogate_slope'):
        if not math.isfinite(cfg[key]):
            raise ValueError(f'유한한 동역학 설정이 필요하다: {key}')
    if (cfg['spike_threshold_radius'] < 0 or cfg['surrogate_slope'] <= 0
            or cfg['spike_threshold_center'] - cfg['spike_threshold_radius'] <= max(0, cfg['spike_reset'])):
        raise ValueError('스파이크 역치는 항상 0과 리셋 전압보다 높아야 한다.')
    if (type(cfg['refractory_ticks']) is not int or cfg['refractory_ticks'] < 0
            or type(cfg['brain_steps']) is not int or cfg['brain_steps'] < 1):
        raise ValueError('불응기는 0 이상의 정수, 토큰당 tick 수는 양의 정수여야 한다.')


class VisualPenLM(nn.Module):
    def __init__(self, cfg, rowptr, src, counts, edge_sign, inputs, outputs, type_ids, dynamics_ids, uv, lamina_nodes, device):
        super().__init__()
        validate_dynamics_config(cfg)
        self.tbptt = cfg['tbptt']
        self.reset_voltage = cfg['spike_reset']
        self.refractory_ticks = cfg['refractory_ticks']
        self.surrogate_slope = cfg['surrogate_slope']
        self.spike_threshold_center = cfg['spike_threshold_center']
        self.spike_threshold_radius = cfg['spike_threshold_radius']
        codes = np.asarray(dynamics_ids)
        if (codes.shape != (cfg['n_neurons'],) or codes.dtype != np.uint8
                or not np.isin(codes, [0, 1, 2]).all()):
            raise ValueError('뉴런 순서에 맞는 uint8 활동 코드 0/1/2가 필요하다.')
        mask = (codes == 2) & (cfg['neuron_mode'] == 'mixed')
        self.register_buffer('spiking_mask', torch.tensor(mask, dtype=torch.bool, device=device), persistent=False)
        self.n, self.k = cfg['n_neurons'], cfg['brain_steps']
        self.block, self.checkpoint_tokens = cfg['edge_block'], cfg['checkpoint_tokens']
        for name, value in [('rowptr', rowptr), ('src', src), ('inputs', inputs),
                            ('outputs', outputs), ('type_ids', type_ids)]:
            self.register_buffer(name, torch.tensor(np.asarray(value), dtype=torch.long,
                                                   device=device), persistent=False)
        assert len(type_ids) == self.n and min(type_ids) >= 0
        self.register_buffer('dst', torch.repeat_interleave(
            torch.arange(self.n, device=device), self.rowptr.diff()), persistent=False)
        sign = torch.tensor(np.asarray(edge_sign), dtype=torch.float32, device=device)
        self.register_buffer('edge_sign', sign, persistent=False)
        c = torch.tensor(np.asarray(counts), dtype=torch.float32, device=device).log1p()
        incoming = torch.zeros(self.n, device=device).index_add_(0, self.dst, c)
        self.register_buffer('prior', c / incoming[self.dst].clamp_min(1e-8), persistent=False)
        raw = torch.where(sign != 0, torch.full_like(sign, math.log(0.2 / 0.8)),
                          torch.where(torch.rand_like(sign) < 0.5, -0.2554128, 0.2554128))
        self.theta = nn.Parameter(raw)
        ntypes = int(max(type_ids)) + 1
        low, high, initial = [torch.tensor(cfg[key], device=device)[:, None]
                              for key in ['tau_min', 'tau_max', 'tau_initial']]
        assert torch.all((low > 0) & (initial > low) & (initial < high))
        self.register_buffer('tau_low', low)
        self.register_buffer('tau_high', high)
        fraction = (initial - low) / (high - low)
        self.raw_tau = nn.Parameter(torch.logit(fraction).expand(3, ntypes).clone())
        self.threshold = nn.Parameter(torch.zeros(ntypes, device=device))
        self.raw_adaptation = nn.Parameter(torch.full((ntypes,), math.log(0.1 / 0.9), device=device))
        self.cfg = dict(cfg)
        self.register_buffer('retina_grid', torch.tensor(uv*2-1,device=device).view(1,1,-1,2))
        self.register_buffer('lamina_nodes',torch.tensor(lamina_nodes,device=device,dtype=torch.long))
        self.register_buffer('joint_low',torch.tensor(cfg['joint_low'],device=device))
        self.register_buffer('joint_high',torch.tensor(cfg['joint_high'],device=device))
        self.motor_norm = nn.LayerNorm(len(outputs),eps=1e-6,elementwise_affine=False,device=device)
        self.pen_head = nn.Linear(len(outputs),cfg['pen_steps']*3,bias=True,device=device)
        nn.init.normal_(self.pen_head.weight,std=0.01)
        nn.init.zeros_(self.pen_head.bias)

    def effective_weights(self):
        magnitude = 0.25 + 3.75 * self.theta.sigmoid()
        signed = torch.where(self.edge_sign != 0, self.edge_sign * magnitude,
                             4 * self.theta.tanh())
        return self.prior * signed

    def dynamics(self):
        tau = self.tau_low + (self.tau_high - self.tau_low) * self.raw_tau.sigmoid()
        alpha = -torch.expm1(-1 / tau[:, self.type_ids])
        offset = self.threshold.tanh()[self.type_ids]
        threshold = torch.where(self.spiking_mask,
            self.spike_threshold_center + self.spike_threshold_radius * offset, 0.5 * offset)
        adaptation = self.raw_adaptation.sigmoid()[self.type_ids]
        return alpha, threshold, adaptation

    def initial_state(self, batch):
        return (self.theta.new_zeros(4,batch,self.n),
                self.theta.new_zeros(batch,len(self.inputs)))

    def sample_retina(self, images):
        if images.ndim != 4 or images.shape[1] != 1 or not images.is_floating_point():
            raise ValueError('Only floating-point grayscale images [B,1,H,W] enter the model')
        # Linearize the screen before spatial sampling.
        linear = torch.where(images <= .04045, images/12.92, ((images+.055)/1.055).pow(2.4))
        blur = self.cfg['retinal_blur']
        linear = F.avg_pool2d(linear,kernel_size=blur,stride=1,padding=blur//2)
        return F.grid_sample(linear,self.retina_grid.expand(len(images),-1,-1,-1),
            mode='bilinear',align_corners=True).reshape(len(images),-1)

    def external_drive(self, luminance):
        value = self.cfg['retinal_gain']*luminance/(self.cfg['retinal_half_saturation']+luminance)
        return luminance.new_zeros(len(luminance),self.n).index_add(1,self.inputs,value)

    def advance(self, image, neural, retina, weights, alpha, threshold, adaptation):
        v,signal,slow,refractory = neural.unbind(0)
        luminance = self.sample_retina(image)
        eye_alpha = 1-math.exp(-1/self.cfg['retina_tau_ticks'])
        intrinsic = torch.zeros_like(v)
        intrinsic[:,self.lamina_nodes] = self.cfg['lamina_rest']
        for _ in range(self.k):
            retina = retina + eye_alpha*(luminance-retina)
            message = SparseEdges.apply(signal,weights,self.rowptr,self.src,self.dst,self.block)
            v,signal,slow,refractory,_ = mixed_neuron_tick(v,signal,slow,refractory,
                message+self.external_drive(retina)+intrinsic,alpha,threshold,adaptation,
                self.spiking_mask,self.reset_voltage,self.refractory_ticks,self.surrogate_slope)
        return torch.stack((v,signal,slow,refractory)),retina

    def decode_pen(self, neural):
        motor = self.motor_norm(neural[1].index_select(1,self.outputs))
        raw = self.pen_head(motor).reshape(len(motor),self.cfg['pen_steps'],3)
        angles = self.joint_low+(self.joint_high-self.joint_low)*raw[...,:2].sigmoid()
        xy = forward_kinematics(angles,self.cfg['link_lengths'])
        return xy,raw[...,2],angles

    def forward(self, images, state=None):
        if images.ndim != 5 or images.shape[2] != 1 or images.shape[1] == 0 or not images.is_floating_point():
            raise ValueError('Expected float image sequence [B,T,1,H,W], never character IDs')
        neural,retina = self.initial_state(len(images)) if state is None else state
        assert neural.shape == (4,len(images),self.n)
        assert retina.shape == (len(images),len(self.inputs))
        weights = self.effective_weights(); alpha,threshold,adaptation = self.dynamics()
        positions,contacts,angles = [],[],[]
        for t in range(images.shape[1]):
            args = (images[:,t],neural,retina,weights,alpha,threshold,adaptation)
            if self.training and self.checkpoint_tokens and torch.is_grad_enabled():
                neural,retina = checkpoint(self.advance,*args,use_reentrant=False,preserve_rng_state=False)
            else:
                neural,retina = self.advance(*args)
            xy,ink,q = self.decode_pen(neural)
            positions.append(xy); contacts.append(ink); angles.append(q)
        return (torch.stack(positions,1),torch.stack(contacts,1),torch.stack(angles,1)),(neural,retina)

def make_model():
    seed_all(CFG['seed'])
    return VisualPenLM(CFG,crow,src,counts,edge_sign,input_nodes,output_nodes,type_ids,
        dynamics_ids,uv,lamina_nodes,device)

def detach_state(state):
    return tuple(x.detach() for x in state)

def template_logits(xy, ink_logits):
    # B*T by C only; no B*T*C*S intermediate allocation.
    a=xy.flatten(-2); b=target_trace[...,:2].flatten(-2)
    distance=(a.square().mean(-1,keepdim=True)+b.square().mean(-1)
              -2*(a@b.T)/a.shape[-1]).clamp_min(0)
    ink=ink_logits.sigmoid(); template_ink=target_trace[...,2]
    distance=distance+CFG['ink_loss_weight']*(ink.square().mean(-1,keepdim=True)+
        template_ink.square().mean(-1)-2*(ink@template_ink.T)/ink.shape[-1]).clamp_min(0)
    return -distance/CFG['template_temperature']

def objective(prediction, labels):
    xy,ink,q=prediction; target=target_trace[labels]
    pos=F.mse_loss(xy,target[...,:2])
    contact=F.binary_cross_entropy_with_logits(ink,target[...,2])
    logits=template_logits(xy,ink)
    nll=F.cross_entropy(logits.flatten(0,1),labels.flatten())
    loss=pos+CFG['ink_loss_weight']*contact+CFG['language_loss_weight']*nll
    metrics=dict(loss=float(loss.detach()), tip_mse=float(pos.detach()),
        contact_bce=float(contact.detach()), template_nll=float(nll.detach()),
        char_accuracy=float((logits.argmax(-1)==labels).float().mean().detach()))
    return loss,metrics

## 7. Numerical and causal checks

Synthetic circuits test sparse gradients, image causality and motor isolation; they never replace
MaleCNS. The full-graph preflight below runs the configured batch/TBPTT with Adam.

In [ ]:
def small_checks(test_device):
    seed_all(123)
    row=torch.tensor([0,1,2,4],device=test_device)
    col=torch.tensor([2,0,0,1],device=test_device)
    dst=torch.tensor([0,1,2,2],device=test_device)
    h=torch.randn(2,3,device=test_device,requires_grad=True)
    w=torch.randn(4,device=test_device,requires_grad=True)
    actual=SparseEdges.apply(h,w,row,col,dst,2)
    matrix=torch.zeros(3,3,device=test_device).index_put((dst,col),w)
    expected=h@matrix.T
    torch.testing.assert_close(actual,expected)
    grad=torch.randn_like(actual)
    ga=torch.autograd.grad(actual,(h,w),grad,retain_graph=True)
    ge=torch.autograd.grad(expected,(h,w),grad)
    for a,b in zip(ga,ge): torch.testing.assert_close(a,b)
    cfg=dict(CFG,n_neurons=6,brain_steps=8,neuron_mode='graded_control',lamina_rest=0.)
    model=VisualPenLM(cfg,np.array([0,0,0,1,2,3,4]),np.array([0,1,2,3]),
        np.ones(4),np.ones(4),np.array([0,1]),np.array([4,5]),np.arange(6),
        np.ones(6,dtype=np.uint8),np.array([[.2,.5],[.8,.5]],np.float32),
        np.array([],dtype=np.int64),test_device)
    image=torch.zeros(1,2,1,12,12,device=test_device)
    image[:,:,:,:,:6]=.8; image.requires_grad_(); model.eval()
    pred,state=model(image); pred2,state2=model(image.flip(-1))
    drive=model.external_drive(torch.ones(1,2,device=test_device))
    assert torch.count_nonzero(drive[:,2:])==0
    assert not torch.allclose(pred[0],pred2[0]), 'Images do not affect the pen'
    (pred[0].square().mean()+pred[1].square().mean()).backward()
    assert image.grad is not None and image.grad.abs().sum()>0
    assert model.theta.grad is not None and model.theta.grad.abs().sum()>0
    s1,s2=state[0].clone(),state2[0].clone()
    s1[1,:,model.outputs]=0; s2[1,:,model.outputs]=0
    torch.testing.assert_close(model.decode_pen(s1)[0],model.decode_pen(s2)[0])
    try: model(torch.zeros(1,2,dtype=torch.long,device=test_device))
    except ValueError: pass
    else: raise AssertionError('Model accepted character IDs')
    z=torch.zeros(1,1,device=test_device); mask=torch.ones_like(z,dtype=torch.bool)
    args=(torch.ones(3,1,device=test_device),torch.full_like(z,.25),z,mask,0.,2,5.)
    v,s,a,r,sp=mixed_neuron_tick(z,z,z,z,torch.ones_like(z),*args)
    assert sp.item()==1 and r.item()==2 and v.item()==0
    *_,blocked=mixed_neuron_tick(v,s,a,r,torch.ones_like(z),*args)
    assert blocked.item()==0
    model.zero_grad(set_to_none=True); model.train(); model.checkpoint_tokens=False
    plain,_=model(image.detach()); plain[0].sum().backward(); gp=model.theta.grad.clone()
    model.zero_grad(set_to_none=True); model.checkpoint_tokens=True
    checked,_=model(image.detach()); checked[0].sum().backward()
    torch.testing.assert_close(plain[0],checked[0]); torch.testing.assert_close(gp,model.theta.grad)
    print('PASS sparse, retinal, motor, refractory, checkpoint checks:',test_device)
small_checks(torch.device('cpu'))
if device.type=='cuda': small_checks(device)
with torch.no_grad():
    canonical_ink=torch.logit(target_trace[...,2].clamp(.00001,.99999))
    recognized=template_logits(target_trace[...,:2],canonical_ink).argmax(-1)
    assert torch.equal(recognized,torch.arange(len(chars),device=device)), 'Canonical traces must self-recognize'
print('PASS: every canonical pen trace self-recognizes')

## 8. Contiguous streams and full-graph preflight

Display t, supervise t+1. Carry detached state through adjacent TBPTT chunks. Reset on wrap;
never train last-to-first or train-to-validation pairs. The disposable preflight checks actual
Adam memory, distinct retinal images and a blank-retina intervention. Fail on silent paths or
exceeded memory budget, without pruning. Full-graph gradient training may be slow even if it fits.

In [ ]:
class CharacterStreams:
    def __init__(self, ids, batch, length):
        self.batch,self.length=batch,length; self.lane_len=len(ids)//batch
        if self.lane_len<length+1: raise ValueError('Dataset too short for batch/TBPTT')
        self.lanes=ids[:batch*self.lane_len].reshape(batch,self.lane_len)
        self.cursor=0; self.epoch=0
    def next(self):
        reset=self.cursor==0
        if self.cursor+self.length>=self.lane_len:
            self.cursor=0; self.epoch+=1; reset=True
        start=self.cursor; self.cursor+=self.length
        x=torch.tensor(self.lanes[:,start:start+self.length],device=device)
        y=torch.tensor(self.lanes[:,start+1:start+self.length+1],device=device)
        return glyph_images[x],y,reset
    def state_dict(self): return dict(cursor=self.cursor,epoch=self.epoch)
    def load_state_dict(self,d):
        if not 0<=d['cursor']<self.lane_len: raise ValueError('Invalid stream position')
        self.cursor=d['cursor']; self.epoch=d['epoch']

def full_preflight():
    gc.collect()
    if device.type=='cuda': torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    test=make_model()
    with torch.no_grad():
        samples=test.sample_retina(glyph_images)
        assert torch.isfinite(samples).all()
        distances=torch.cdist(samples.double(),samples.double()); distances.fill_diagonal_(float('inf'))
        assert distances.min()>1e-5, 'Two glyphs are indistinguishable at the retina'
        print('Minimum retinal inter-glyph distance:',float(distances.min()))
    stream=CharacterStreams(train_ids,CFG['batch_size'],CFG['tbptt'])
    images,labels,_=stream.next()
    opt=torch.optim.Adam(test.parameters(),lr=CFG['lr'],foreach=False)
    start=time.perf_counter(); prediction,state=test(images)
    loss,metrics=objective(prediction,labels); assert torch.isfinite(loss)
    loss.backward(); theta_norm=float(test.theta.grad.norm())
    assert theta_norm>0 and math.isfinite(theta_norm), 'No finite synaptic gradient'
    torch.nn.utils.clip_grad_norm_(test.parameters(),1.,error_if_nonfinite=True); opt.step()
    test.eval()
    with torch.no_grad():
        real,sreal=test(images); blank,sblank=test(torch.zeros_like(images))
        delta=float((real[0]-blank[0]).abs().max())
        motor_delta=float((sreal[0][1,:,test.outputs]-sblank[0][1,:,test.outputs]).abs().max())
        assert delta>1e-8 and motor_delta>1e-10, 'Visual-motor path silent: inspect dynamics/ticks'
    if device.type=='cuda': torch.cuda.synchronize()
    peak=torch.cuda.max_memory_allocated()/2**30 if device.type=='cuda' else None
    result=dict(metrics,seconds=time.perf_counter()-start,peak_allocated_GiB=peak,
        edge_gradient_norm=theta_norm,blank_retina_pen_delta=delta,motor_delta=motor_delta)
    if device.type=='cuda':
        budget=CFG['vram_fraction']*torch.cuda.get_device_properties(device).total_memory/2**30
        assert peak<budget, 'GPU budget exceeded; lower batch_size/tbptt and rerun from the start'
    del test,opt,prediction,state,loss,real,blank,sreal,sblank
    gc.collect()
    if device.type=='cuda': torch.cuda.empty_cache()
    seed_all(CFG['seed']); return result
PREFLIGHT=full_preflight()
(RUN_DIR/'preflight.json').write_text(json.dumps(PREFLIGHT,indent=2),encoding='utf-8')
print('PASS full-graph preflight:',PREFLIGHT)

## 9. Validation, checkpoints and portable assets

Save graph arrays, source hashes, visual templates, port IDs and experimental settings.
Resume restores model, Adam, stream position, detached neural/retinal state and RNG at a TBPTT
boundary. It requires matching graph/config/visual content and this notebook's definitions.
CUDA scatter reductions may be nondeterministic; this is not a bitwise GPU reproducibility claim.
Only load your own trusted checkpoints; optimizer/RNG restore uses `weights_only=False`.

In [ ]:
def save_array_assets():
    graph_dir=RUN_DIR/'graph'; graph_dir.mkdir(exist_ok=True)
    arrays=dict(body_ids=body_ids,crow=crow,src=src,counts=counts,edge_sign=edge_sign,
        input_nodes=input_nodes,output_nodes=output_nodes,type_ids=type_ids,dynamics_ids=dynamics_ids,
        uv=uv,lamina_nodes=lamina_nodes)
    fingerprints={}
    for key,value in arrays.items():
        np.save(graph_dir/(key+'.npy'),value)
        fingerprints[key]=sha256_file(graph_dir/(key+'.npy'))
    torch.save(dict(images=glyph_images_cpu,trace=target_trace_cpu,joints=target_joints_cpu,chars=chars),
        RUN_DIR/'visual_pen_assets.pt')
    spec=dict(format='flylm3_visual_foreleg_v1',cfg=CFG,data=DATA_SPEC,font=FONT_SPEC,
        graph=GRAPH_SPEC,sources=source_manifest,dynamics=dynamics_spec,versions=VERSIONS)
    (RUN_DIR/'experiment.json').write_text(json.dumps(spec,ensure_ascii=False,indent=2),encoding='utf-8')
    semantic=dict(cfg=CFG,data=DATA_SPEC,dynamics=dynamics_spec,arrays=fingerprints,
        image_sha=hashlib.sha256(glyph_images_cpu.numpy().tobytes()).hexdigest(),
        trace_sha=hashlib.sha256(target_trace_cpu.numpy().tobytes()).hexdigest())
    (RUN_DIR/'asset_manifest.json').write_text(json.dumps(semantic,indent=2),encoding='utf-8')
    return hashlib.sha256(json.dumps(semantic,sort_keys=True).encode()).hexdigest()

@torch.no_grad()
def evaluate(model, blank_retina=False):
    previous=model.training; model.eval()
    stream=CharacterStreams(val_ids,CFG['batch_size'],CFG['tbptt'])
    state=None; rows=[]
    for _ in range(CFG['eval_chunks']):
        images,labels,reset=stream.next()
        if reset: state=None
        if blank_retina: images=torch.zeros_like(images)
        pred,state=model(images,state)
        _,metrics=objective(pred,labels); rows.append(metrics)
    model.train(previous)
    return {key:float(np.mean([r[key] for r in rows])) for key in rows[0]}

def save_checkpoint(path,model,optimizer,stream,state,update,best):
    payload=dict(format='flylm3_visual_foreleg_v1',identity=EXPERIMENT_ID,
        model=model.state_dict(),optimizer=optimizer.state_dict(),stream=stream.state_dict(),
        neural_state=None if state is None else tuple(x.detach().cpu() for x in state),
        update=update,best=best,torch_rng=torch.get_rng_state(),
        cuda_rng=torch.cuda.get_rng_state_all() if device.type=='cuda' else None,
        numpy_rng=np.random.get_state(),python_rng=random.getstate())
    partial=path.with_suffix('.part'); torch.save(payload,partial); partial.replace(path)

def restore_checkpoint(path,model,optimizer,stream):
    saved=torch.load(path,map_location='cpu',weights_only=False)
    if saved.get('format')!='flylm3_visual_foreleg_v1' or saved.get('identity')!=EXPERIMENT_ID:
        raise ValueError('Checkpoint/config/data/graph/assets mismatch; cannot resume')
    model.load_state_dict(saved['model']); optimizer.load_state_dict(saved['optimizer'])
    stream.load_state_dict(saved['stream'])
    state=None if saved['neural_state'] is None else tuple(x.to(device) for x in saved['neural_state'])
    torch.set_rng_state(saved['torch_rng'])
    if device.type=='cuda' and saved['cuda_rng'] is not None: torch.cuda.set_rng_state_all(saved['cuda_rng'])
    np.random.set_state(saved['numpy_rng']); random.setstate(saved['python_rng'])
    return state,int(saved['update']),float(saved['best'])

## 10. Train next-character pen prediction

Set `RESUME_FROM` before running this cell to continue a matching previous run. Otherwise start
fresh. Validation covers a fixed prefix in each held-out lane (`eval_chunks × batch × tbptt`
characters); increase coverage for reporting. Save best by validation loss and latest periodically.
The checkpoint records the stream state at an optimizer boundary, as in ordinary stateful TBPTT.
Logs include elapsed seconds and ETA (`HH:MM:SS`), estimated from updates completed in this session.


In [ ]:
RESUME_FROM = None  # Path('/kaggle/input/my-run/latest.pt')
EXPERIMENT_ID=save_array_assets()
model=make_model()
optimizer=torch.optim.Adam(model.parameters(),lr=CFG['lr'],foreach=False)
train_stream=CharacterStreams(train_ids,CFG['batch_size'],CFG['tbptt'])
state=None; start_update=0; best=float('inf')
if RESUME_FROM is not None:
    state,start_update,best=restore_checkpoint(Path(RESUME_FROM),model,optimizer,train_stream)
print('Parameters:',sum(p.numel() for p in model.parameters()),'resume update:',start_update)
initial_eval=evaluate(model); print('Initial validation:',initial_eval)
start_time=time.perf_counter(); last_update=start_update
for update in range(start_update+1,CFG['train_updates']+1):
    model.train(); images,labels,reset=train_stream.next()
    if reset: state=None
    optimizer.zero_grad(set_to_none=True)
    prediction,state=model(images,state)
    loss,metrics=objective(prediction,labels)
    if not torch.isfinite(loss): raise FloatingPointError(f'Nonfinite loss at {update}')
    loss.backward()
    grad_norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.,error_if_nonfinite=True)
    optimizer.step(); state=detach_state(state); last_update=update
    elapsed_seconds=time.perf_counter()-start_time
    eta_seconds=elapsed_seconds/(update-start_update)*max(0,CFG['train_updates']-update)
    eta_minutes,eta_secs=divmod(math.ceil(eta_seconds),60)
    eta_hours,eta_minutes=divmod(eta_minutes,60)
    row=dict(update=update,epoch=train_stream.epoch,**metrics,grad_norm=float(grad_norm),
        characters_per_second=(update-start_update)*CFG['batch_size']*CFG['tbptt']/elapsed_seconds,
        elapsed_seconds=elapsed_seconds,eta_seconds=eta_seconds,
        eta=f'{eta_hours:02d}:{eta_minutes:02d}:{eta_secs:02d}')
    if update%CFG['log_every']==0 or update==start_update+1 or update==CFG['train_updates']: print(row)
    if update%CFG['eval_every']==0 or update==CFG['train_updates']:
        val=evaluate(model); row['validation']=val; print('Validation:',val)
        if val['loss']<best:
            best=val['loss']; save_checkpoint(RUN_DIR/'best.pt',model,optimizer,train_stream,state,update,best)
    if update%CFG['save_every']==0 or update==CFG['train_updates']:
        save_checkpoint(RUN_DIR/'latest.pt',model,optimizer,train_stream,state,update,best)
    with (RUN_DIR/'metrics.jsonl').open('a',encoding='utf-8') as f: f.write(json.dumps(row)+'\n')
    del loss,prediction
save_checkpoint(RUN_DIR/'latest.pt',model,optimizer,train_stream,state,last_update,best)
print('Completed:',last_update,'updates; artifacts:',RUN_DIR)

## 10A. Save run as ZIP

Run after training to download checkpoints, graph arrays, visual/pen assets, and run metadata together.
The archive is saved beside the run folder; rerun this cell after step 11 to include generated previews.


In [ ]:
from pathlib import Path
import shutil
from IPython.display import FileLink, display

# Save the complete run, including best.pt and latest.pt, as in v2's ZIP export.
run_dir = Path(RUN_DIR).resolve()
for name in ('latest.pt', 'visual_pen_assets.pt', 'experiment.json', 'asset_manifest.json'):
    if not (run_dir / name).is_file():
        raise FileNotFoundError(f'Missing run artifact: {run_dir / name}. Run step 10 first.')
if not (run_dir / 'graph').is_dir():
    raise FileNotFoundError(f'Missing graph directory: {run_dir / "graph"}')

# Keep the archive outside the run directory so repeated exports cannot include themselves.
zip_path = Path(shutil.make_archive(
    str(run_dir.parent / f'{run_dir.name}_flylm3'),
    'zip', root_dir=run_dir.parent, base_dir=run_dir.name,
))
print('Saved:', zip_path)
print('Download this ZIP from Kaggle Files. Keep the extracted run directory together.')
try:
    display(FileLink(str(zip_path.relative_to(Path.cwd().resolve()))))
except ValueError:
    print('The ZIP is outside the notebook directory; use the path above to download it.')


## 11. Generate actual pen traces and animate the leg

Process prompt images once. Rasterize each predicted trace and feed **its actual ink** back to
R1–R6. Nearest-template character labels are diagnostics only, never clean replacement images.
Feedback may drift from teacher-forced training; the panels and animation expose the produced
strokes. Whitespace control marks remain visible. The pen's first sample is a pen-up positioning
move, and only subsequent samples with contact draw ink.

In [ ]:
@torch.no_grad()
def generate_pen(model,prompt='ROMEO:',new_characters=24):
    if not prompt or any(ch not in char_to_index for ch in prompt):
        raise ValueError('Use a nonempty prompt from the saved character alphabet')
    if new_characters<1: raise ValueError('new_characters must be positive')
    previous=model.training; model.eval(); state=None
    for ch in prompt:
        frame=glyph_images[char_to_index[ch]][None,None]
        prediction,state=model(frame,state)
    traces=[]; joints=[]; recognized=[]; margins=[]
    for _ in range(new_characters):
        xy,ink,q=prediction
        trace=torch.cat([xy[0,0],ink[0,0].sigmoid()[:,None]],dim=-1).cpu().numpy()
        logits=template_logits(xy,ink)[0,0]; top=logits.topk(2)
        recognized.append(chars[int(top.indices[0])]); margins.append(float(top.values[0]-top.values[1]))
        traces.append(trace); joints.append(q[0,0].cpu().numpy())
        observed=torch.from_numpy(rasterize_trace(trace)).to(device)[None,None,None]
        prediction,state=model(observed,state)
    model.train(previous)
    return dict(prompt=prompt,text=''.join(recognized),traces=np.stack(traces),
        joints=np.stack(joints),template_margins=margins)

sample=generate_pen(model)
print('Prompt:',repr(sample['prompt']))
print('Recognized pen output (diagnostic):',repr(sample['text']))
fig,axes=plt.subplots(3,8,figsize=(12,5))
for ax,trace,ch in zip(axes.flat,sample['traces'],sample['text']):
    ax.imshow(rasterize_trace(trace),cmap='gray',vmin=0,vmax=1)
    ax.set_title(repr(ch)); ax.axis('off')
plt.tight_layout(); fig.savefig(RUN_DIR/'generated_pen.png',dpi=150); plt.show()
np.savez(RUN_DIR/'generated_pen.npz',traces=sample['traces'],joints=sample['joints'])
(RUN_DIR/'generated_pen.json').write_text(json.dumps({k:v for k,v in sample.items()
    if k not in ['traces','joints']},ensure_ascii=False,indent=2),encoding='utf-8')

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

def animate_leg(trace,joints):
    fig,ax=plt.subplots(figsize=(5,5)); ax.set_aspect('equal')
    ax.set_xlim(-.1,1.7); ax.set_ylim(-1.,1.)
    ax.set_title('Right-foreleg kinematic surrogate')
    links,=ax.plot([],[],'o-',lw=2); inkline,=ax.plot([],[],color='black',lw=1)
    def draw(k):
        q1,q2=joints[k]; l1,l2=CFG['link_lengths']
        elbow=np.array([l1*np.cos(q1),l1*np.sin(q1)])
        tip=trace[k,:2]; links.set_data([0,elbow[0],tip[0]],[0,elbow[1],tip[1]])
        xs=[]; ys=[]
        for j in range(1,k+1):
            if trace[j,2]>=.5:
                xs.extend([trace[j-1,0],trace[j,0],np.nan]); ys.extend([trace[j-1,1],trace[j,1],np.nan])
        inkline.set_data(xs,ys); return links,inkline
    anim=FuncAnimation(fig,draw,frames=len(trace),interval=35,blit=True)
    html=anim.to_jshtml(); plt.close(fig); return html
animation_html=animate_leg(sample['traces'][0],sample['joints'][0])
display(HTML(animation_html))
(RUN_DIR/'leg_pen_animation.html').write_text(animation_html,encoding='utf-8')

## 12. Controls and report

Compare held-out metrics to blank-retina and clamped-foreleg interventions, and train-only
unigram NLL on the same labels. Template NLL is an engineered distribution, not conventional
LM perplexity. Low trajectory error may reflect averaged, unreadable predictions. For a mixed
versus graded control, rerun with matched seed, corpus, exposure and simulation ticks.

In [ ]:
normal=evaluate(model); blind=evaluate(model,blank_retina=True)
hook=model.motor_norm.register_forward_pre_hook(lambda module,args:(torch.zeros_like(args[0]),))
try: motor_clamped=evaluate(model)
finally: hook.remove()
frequencies=np.bincount(train_ids,minlength=len(chars)).astype(np.float64)+1
frequencies/=frequencies.sum()
es=CharacterStreams(val_ids,CFG['batch_size'],CFG['tbptt']); unigram=[]
for _ in range(CFG['eval_chunks']):
    _,labels,_=es.next(); unigram.extend(labels.cpu().numpy().ravel().tolist())
unigram_nll=float(-np.log(frequencies[np.array(unigram)]).mean())
REPORT=dict(initial_validation=initial_eval,trained=normal,blank_retina=blind,
    foreleg_clamped=motor_clamped,train_only_unigram_nll=unigram_nll,
    validation_characters=CFG['eval_chunks']*CFG['batch_size']*CFG['tbptt'],
    limitation='Prefix validation; template likelihood; open-loop kinematics; no physiological claim')
(RUN_DIR/'evaluation.json').write_text(json.dumps(REPORT,indent=2),encoding='utf-8')
display(pd.DataFrame({k:v for k,v in REPORT.items() if isinstance(v,dict)}).T)
print('Unigram NLL on identical labels:',unigram_nll)
print('Preserve notebook and run directory before hosted runtime expires:',RUN_DIR)

## Sources and correspondence

- [nanoGPT character preparation](https://github.com/Lucid-Frameworks/nanoGPT/blob/master/data/shakespeare_char/prepare.py): raw Tiny Shakespeare and 90/10 split. The linked `shakespeare` directory uses a different preparation; this follows the requested character-level variant.
- [doomfly](https://github.com/nftechie/doomfly/blob/main/doom/README.md), [retinal projection](https://github.com/nftechie/doomfly/blob/main/doom/prepare.py), [engine](https://github.com/nftechie/doomfly/blob/main/doom/engine.py): retinal projection, luminance filtering and saturation adapted to differentiable v2 dynamics, not a reproduction of its LIF timing or Doom task.
- [MaleCNS release](https://male-cns.janelia.org/download/): actual connectivity, receptor columns, transmitters and right-foreleg neuron IDs.
- [Azevedo et al., female ventral nerve cord reconstruction](https://pmc.ncbi.nlm.nih.gov/articles/PMC11348827/), [DOI](https://doi.org/10.1038/s41586-024-07389-x): FANC motor anatomy. Different specimen; IDs are not merged with MaleCNS.
- [Lesser et al., leg and wing premotor networks](https://www.nature.com/articles/s41586-024-07600-z), [author-hosted record](https://authors.library.caltech.edu/records/ekf21-21m53): motor organization motivates a restricted leg readout. It supplies neither pen physics nor a writing controller.
- [Connectome-constrained visual modeling](https://doi.org/10.1038/s41586-024-07939-3): graded visual modeling precedent.
- [PyTorch builds](https://pytorch.org/get-started/previous-versions/): CUDA 12.6 setup.

These documents are research references, not instructions changing the user's request.
The inherited activity table cites its electrophysiology sources beside individual assignments.
The joint limits, links, pen head, targets, lamina resting offset and template likelihood are
engineering choices. No muscles, contact forces or proprioception are modeled. A learned output
prior can operate without vision, so the interventions are essential to interpreting results.

### doomfly license attribution

MIT License

Copyright (c) 2026 nftechie and DOOMFLY contributors

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

## Authoring validation — 2026-09-10

The notebook was executed top to bottom on an RTX 4060 Ti (8 GiB), PyTorch 2.10.0+cu126,
with only `train_updates` shortened from 1,000 to 2 and output paths redirected to a test folder.
The full retained graph had 165,122 neurons, 25,563,197 edges, 1,386 mapped R1–R6 receptors
and 67 right-foreleg motor neurons; all selected motors were reachable from the retina.
The configured batch 2 / TBPTT 16 / 16 ticks preflight passed real Adam/backpropagation,
nonzero edge gradients and a blank-retina intervention, using ~1.47 GiB peak allocated GPU memory.
CPU/CUDA numerical checks, synthetic stream/resume tests, full checkpoint reload, trace generation,
leg animation and ablation reporting passed. Glyph sheets were visually inspected.

This is an implementation smoke test, not a completed training result or a P100 benchmark.
Two-update generation was not readable text and did not beat the unigram baseline.
No trained weights or test outputs are embedded here. All code remains in this one notebook;
running it creates the documented data caches and run artifacts. The v2 notebook is unchanged.